<a href="https://colab.research.google.com/github/NamishBansal15/substation-detection/blob/main/model-training/cascade_resnet50_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# INSTALL ALL DEPENDENCIES — run this first
# ═══════════════════════════════════════════════════════════════════════════
import os
os.chdir('/content')

# Detectron2 (matches your CUDA/PyTorch from before)
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

# Roboflow
!pip install -q roboflow

# Verify
import detectron2
import roboflow
print(f'Detectron2: {detectron2.__version__}')
print('Roboflow installed.')
print('All dependencies ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CASCADE R-CNN RESNET-50 FPN — COMPLETE PIPELINE IN ONE CELL
# Saves directly to Drive during training so disconnects can't lose progress
# ═══════════════════════════════════════════════════════════════════════════

import os, json, glob, getpass, csv
from google.colab import drive
from roboflow import Roboflow
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog, build_detection_test_loader
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

# ── 1. Mount Drive ────────────────────────────────────────────────────────
drive.mount('/content/drive')
DRIVE_OUTPUT_DIR = os.environ.get("CASCADE_R50_OUTPUT_DIR", "/content/drive/MyDrive/Cascade_RCNN/ResNet50")
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# ── 2. Download dataset from Roboflow ─────────────────────────────────────
RF_API_KEY   = getpass.getpass('Roboflow API Key: ')
RF_WORKSPACE = input('Workspace name: ').strip()
RF_PROJECT   = input('Project name: ').strip()
RF_VERSION   = int(input('Version number: ').strip())

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = project.version(RF_VERSION)
dataset = version.download('coco', location='/content/dataset')
DATASET_ROOT = '/content/dataset'

# ── 3. Verify dataset + detect classes ────────────────────────────────────
TRAIN_DIR = f'{DATASET_ROOT}/train'
VALID_DIR = f'{DATASET_ROOT}/valid'
TEST_DIR  = f'{DATASET_ROOT}/test'

def find_ann(split_dir):
    for fname in ['_annotations.coco.json', 'annotations.json']:
        path = f'{split_dir}/{fname}'
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f'No annotation file in {split_dir}')

TRAIN_ANN = find_ann(TRAIN_DIR)
VALID_ANN = find_ann(VALID_DIR)
TEST_ANN  = find_ann(TEST_DIR)

with open(TRAIN_ANN) as f:
    coco_data = json.load(f)
CLASSES = [cat['name'] for cat in sorted(coco_data['categories'], key=lambda x: x['id'])]
NUM_CLASSES = len(CLASSES)
print(f'Classes ({NUM_CLASSES}): {CLASSES}')

# ── 4. Register datasets ──────────────────────────────────────────────────
for split in ['train', 'valid', 'test']:
    name = f'substation_{split}'
    if name in DatasetCatalog:
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

register_coco_instances('substation_train', {}, TRAIN_ANN, TRAIN_DIR)
register_coco_instances('substation_valid', {}, VALID_ANN, VALID_DIR)
register_coco_instances('substation_test',  {}, TEST_ANN,  TEST_DIR)
for split in ['train', 'valid', 'test']:
    MetadataCatalog.get(f'substation_{split}').thing_classes = CLASSES

# ── 5. Configure Cascade R-CNN ResNet-50 FPN ──────────────────────────────
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file('Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml'))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url('Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml')

cfg.DATASETS.TRAIN = ('substation_train',)
cfg.DATASETS.TEST  = ('substation_valid',)
cfg.DATALOADER.NUM_WORKERS = 2

cfg.SOLVER.IMS_PER_BATCH     = 4
cfg.SOLVER.BASE_LR           = 0.001
cfg.SOLVER.WARMUP_FACTOR     = 0.001
cfg.SOLVER.MAX_ITER          = 13000
cfg.SOLVER.STEPS             = (9000, 12000)
cfg.SOLVER.GAMMA             = 0.1
cfg.SOLVER.CHECKPOINT_PERIOD = 500
cfg.SOLVER.WARMUP_ITERS      = 500
cfg.TEST.EVAL_PERIOD         = 500

cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
cfg.MODEL.MASK_ON = False

cfg.INPUT.MIN_SIZE_TRAIN = (640,)
cfg.INPUT.MAX_SIZE_TRAIN = 640
cfg.INPUT.MIN_SIZE_TEST  = 640
cfg.INPUT.MAX_SIZE_TEST  = 640

cfg.OUTPUT_DIR = f'{DRIVE_OUTPUT_DIR}/output'
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print(f'Output saving directly to Drive: {cfg.OUTPUT_DIR}')
print(f'Backbone: ResNet-50')
print(f'Max iterations: {cfg.SOLVER.MAX_ITER} (~50 epochs)')

# ── 6. Train ──────────────────────────────────────────────────────────────
class TrainerWithEval(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, 'eval')
        return COCOEvaluator(dataset_name, cfg, False, output_folder)

trainer = TrainerWithEval(cfg)
trainer.resume_or_load(resume=False)
trainer.train()
print('Training complete.')

# ── 7. Load best checkpoint ───────────────────────────────────────────────
ckpts = sorted(glob.glob(f'{cfg.OUTPUT_DIR}/model_*.pth'))
BEST_CKPT = ckpts[-1] if ckpts else os.path.join(cfg.OUTPUT_DIR, 'model_final.pth')
print(f'Using checkpoint: {BEST_CKPT}')

cfg.MODEL.WEIGHTS = BEST_CKPT
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
predictor = DefaultPredictor(cfg)

# ── 8. Evaluate on validation set ─────────────────────────────────────────
print('\n=== Validation Set ===')
val_evaluator = COCOEvaluator('substation_valid', cfg, False, output_dir=f'{cfg.OUTPUT_DIR}/eval_valid')
val_loader = build_detection_test_loader(cfg, 'substation_valid')
val_results = inference_on_dataset(predictor.model, val_loader, val_evaluator)
print(val_results)

# ── 9. Evaluate on test set ───────────────────────────────────────────────
print('\n=== Test Set ===')
test_evaluator = COCOEvaluator('substation_test', cfg, False, output_dir=f'{cfg.OUTPUT_DIR}/eval_test')
test_loader = build_detection_test_loader(cfg, 'substation_test')
test_results = inference_on_dataset(predictor.model, test_loader, test_evaluator)
print(test_results)

# ── 10. Summary table + chart stats ──────────────────────────────────────
MODEL_NAME = 'Cascade R-CNN ResNet-50 FPN'

val_ap50 = val_results['bbox'].get('AP50')
val_ap   = val_results['bbox'].get('AP')
test_ap50 = test_results['bbox'].get('AP50')
test_ap   = test_results['bbox'].get('AP')

per_class = {}
for k, v in test_results['bbox'].items():
    if k.startswith('AP-'):
        per_class[k.replace('AP-', '')] = v

target_classes = ['Alt Energy', 'Circuit Breaker', 'Control', 'Power Lines', 'Reactor', 'Transformer']
panel_labels   = ['(c)', '(d)', '(e)', '(f)', '(g)', '(h)']

print('\n' + '=' * 65)
print(f'  FINAL RESULTS — {MODEL_NAME}')
print('=' * 65)
print(f'\n(a) Overall Performance (mAP@50):')
print(f'    Validation : {val_ap50:.2f}')
print(f'    Test       : {test_ap50:.2f}')
print(f'\n(b) Overall Performance (mAP@50:95):')
print(f'    Validation : {val_ap:.2f}')
print(f'    Test       : {test_ap:.2f}')
print()
for label, cls in zip(panel_labels, target_classes):
    val = per_class.get(cls, float('nan'))
    print(f'{label} {cls} (AP test): {val:.2f}')

print('=' * 65)

# ── 11. Save CSV to Drive ─────────────────────────────────────────────────
csv_path = f'{DRIVE_OUTPUT_DIR}/results_for_charts.csv'
rows = [
    ['Metric', 'Validation', 'Test'],
    ['mAP@50 Overall',    f'{val_ap50:.2f}',  f'{test_ap50:.2f}'],
    ['mAP@50:95 Overall', f'{val_ap:.2f}',    f'{test_ap:.2f}'],
]
for cls in target_classes:
    v = per_class.get(cls, float('nan'))
    rows.append([f'AP - {cls}', 'N/A', f'{v:.2f}'])

with open(csv_path, 'w', newline='') as f:
    csv.writer(f).writerows(rows)

print(f'\nCSV saved to: {csv_path}')
print(f'Checkpoint saved at: {BEST_CKPT}')
print('Everything saved directly to Google Drive.')